# Deep Dive Analysis: Network Markowitz (Proposisi) vs Momentum (2025)

Notebook ini memperluas analisis dengan menambahkan model **Network Markowitz (Proposisi)** yang menggabungkan:
1. **RMT (Random Matrix Theory)**: Membersihkan noise pada matriks kovarians.
2. **MST (Minimum Spanning Tree)**: Menyaring struktur topologi pasar yang penting.
3. **Network Centrality**: Menentukan bobot alokasi berdasarkan sentralitas jaringan.
4. **Variasi Gamma (γ)**: Mengatur keseimbangan antara optimasi Markowitz murni (γ=0) dan Network-Based Allocation (γ=1).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from scipy.optimize import minimize
import networkx as nx
from sklearn.metrics import fbeta_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
sns.set_palette("husl")
SEED = 42
np.random.seed(SEED)

print("✓ Robust Environment Ready!")

## 1. Load Data & Setup

In [2]:
file_path = '../experiment_nextLevel2/dataset_2023_2025.xlsx'
data = pd.read_excel(file_path, index_col=0, parse_dates=True)
returns = data.pct_change().dropna()
market_return = returns.mean(axis=1)
market_index = (1 + market_return).cumprod() * 100

print(f"Data: {len(data)} rows, {len(data.columns)} assets")

## 2. Model Training & Probabilities

In [3]:
features = pd.DataFrame(index=returns.index)
features['Vol_20'] = market_return.rolling(window=20).std()
features['Mom_20'] = market_return.rolling(window=20).mean()
features['Mom_50'] = market_return.rolling(window=50).mean()
target = (market_return.rolling(5).mean() > market_return.rolling(20).mean()).astype(int).shift(-5)

X_train = features.loc[features.index.year <= 2024].dropna()
y_train = target.loc[X_train.index].fillna(0)
xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=SEED)
xgb_model.fit(X_train, y_train)

all_probs = pd.Series(xgb_model.predict_proba(features.dropna())[:, 1], index=features.dropna().index)
all_probs = all_probs.reindex(returns.index, method='ffill').fillna(method='bfill')
opt_t = 0.52
print("✓ AI Probabilities Ready.")

## 3. Robust Simulation Engines (Existing)

In [4]:
def optimize_markowitz_robust(selected_returns, cov_matrix=None):
    clean_returns = selected_returns.dropna(axis=1, how='any')
    if clean_returns.empty: return {}
    if len(clean_returns.columns) == 1: return {clean_returns.columns[0]: 1.0}
    
    mu = clean_returns.mean() * 252
    if cov_matrix is not None:
        sigma = cov_matrix
    else:
        sigma = clean_returns.cov() * 252
        sigma += np.diag(np.ones(len(sigma)) * 1e-4) # Stabilization
    
    num_assets = len(mu)
    def objective(w):
        ret = np.sum(w * mu)
        risk = np.sqrt(np.dot(w.T, np.dot(sigma, w)))
        return -(ret / risk) if risk > 1e-6 else 0
    
    res = minimize(objective, [1./num_assets]*num_assets, method='SLSQP', 
                   bounds=tuple((0, 1) for _ in range(num_assets)), 
                   constraints=({'type': 'eq', 'fun': lambda x: np.sum(x) - 1}))
    
    return dict(zip(clean_returns.columns, res.x)) if res.success else {c: 1./num_assets for c in clean_returns.columns}

def get_assets_graph_selection(returns_window, corr_threshold=0.4, top_n=25, strategy_type='diversify'):
    momentum_assets = returns_window.mean().sort_values(ascending=False).head(top_n).index
    returns_mom = returns_window[momentum_assets]
    corr_mat = returns_mom.corr()
    G = nx.Graph()
    G.add_nodes_from(momentum_assets)
    for i, a1 in enumerate(momentum_assets):
        for a2 in momentum_assets[i+1:]:
            if strategy_type == 'diversify':
                if abs(corr_mat.loc[a1, a2]) > corr_threshold: G.add_edge(a1, a2)
            else:
                if abs(corr_mat.loc[a1, a2]) < corr_threshold: G.add_edge(a1, a2)
    return list(nx.approximation.maximum_independent_set(G))

def run_simulation_ai_momentum(test_dates, returns, ai_probs, threshold, market_idx, top_n, strategy_type, fee=0.0025):
    val = 100.0; history = [val]; dates = [test_dates[0]]
    current_weights = {}; risk_status = False; days_since_rebal = 999
    market_ma = market_idx.rolling(window=200).mean()

    for i, date in enumerate(test_dates[:-1]):
        prob = ai_probs.loc[date]
        prev_risk = risk_status
        if not risk_status and prob > (threshold + 0.05): risk_status = True
        elif risk_status and prob < (threshold - 0.05): risk_status = False
        if market_idx.loc[date] > market_ma.loc[date]: risk_status = True
            
        target_weights = current_weights.copy()
        if not risk_status:
            target_weights = {'CASH': 1.0}; days_since_rebal = 0
        else:
            if (risk_status != prev_risk) or days_since_rebal >= 20:
                try:
                    loc_idx = returns.index.get_loc(date)
                    window = returns.iloc[max(0, loc_idx-60):loc_idx]
                    selected = get_assets_graph_selection(window, top_n=top_n, strategy_type=strategy_type)
                    if selected: target_weights = optimize_markowitz_robust(window[selected]); days_since_rebal = 0
                except: pass
        
        days_since_rebal += 1
        turnover = sum(abs(target_weights.get(k, 0) - current_weights.get(k, 0)) for k in set(target_weights)|set(current_weights))
        val -= val * turnover * fee
        next_date = test_dates[i+1]
        day_ret = 0; new_drifted = {}
        if 'CASH' in target_weights: 
            new_drifted = {'CASH': 1.0}
        else:
            for asset, w in target_weights.items():
                r = returns.loc[next_date, asset] if next_date in returns.index else 0
                day_ret += w * r
                new_drifted[asset] = w * (1 + r)
        val *= (1 + day_ret); history.append(val); dates.append(next_date)
        current_weights = {k: v/sum(new_drifted.values()) for k, v in new_drifted.items()} if sum(new_drifted.values()) > 0 else new_drifted
    return pd.DataFrame({'Portfolio_Value': history}, index=dates)

def run_simulation_top15_static_markowitz(test_dates, returns):
    start_date = test_dates[0]
    loc_idx = returns.index.get_loc(start_date)
    pre_window = returns.iloc[max(0, loc_idx-60):loc_idx]
    
    # 1. Pilih Top 15 Berdasarkan Momentum
    top15_momentum = pre_window.mean().sort_values(ascending=False).head(15).index
    
    # 2. Ambil Bobot Markowitz dari Top 15 Tersebut
    static_weights = optimize_markowitz_robust(pre_window[top15_momentum])
    
    val = 100.0; history = [val]; dates = [start_date]
    for i, date in enumerate(test_dates[:-1]):
        next_date = test_dates[i+1]
        day_ret = sum(w * (returns.loc[next_date, asset] if next_date in returns.index else 0) for asset, w in static_weights.items())
        val *= (1 + day_ret); history.append(val); dates.append(next_date)
    
    print(f"✓ Optimized weights for TOP 15 Static Baseline:")
    print({k: round(v, 3) for k, v in static_weights.items() if v > 0.01})
    return pd.DataFrame({'Portfolio_Value': history}, index=dates)

## 3.5. 🔬 Uji Signifikansi Eigenvalue (Random Matrix Theory)

In [5]:
def plot_rmt_test(returns_window, title="RMT Eigenvalue Significance Test"):
    """
    Memvisualisasikan distribusi eigenvalue empiris vs distribusi teoritis Marcenko-Pastur.
    Tujuannya untuk melihat eigenvalue mana yang 'signal' (di atas lambda_max) dan mana 'noise'.
    """
    T, N = returns_window.shape
    Q = T / N
    corr_mat = returns_window.corr()
    evals, _ = np.linalg.eigh(corr_mat)
    evals = evals[::-1]  # Sort descending

    # Marcenko-Pastur Theoretical PDF
    sigma2 = 1.0 # Normalized returns variance
    lambda_max = sigma2 * (1 + 1.0/Q + 2*np.sqrt(1.0/Q))
    lambda_min = sigma2 * (1 + 1.0/Q - 2*np.sqrt(1.0/Q))
    
    # Generate PDF points
    pdf_x = np.linspace(lambda_min, lambda_max, 1000)
    pdf_y = (Q / (2 * np.pi * sigma2)) * np.sqrt(np.maximum(0, (lambda_max - pdf_x) * (pdf_x - lambda_min))) / pdf_x
    
    # Plotting
    plt.figure(figsize=(12, 6))
    sns.histplot(evals, bins=min(50, len(evals)), stat="density", alpha=0.5, label='Empirical Eigenvalues', kde=True)
    plt.plot(pdf_x, pdf_y, 'r-', linewidth=2, label='Marcenko-Pastur Distribution (Noise)')
    plt.axvline(x=lambda_max, color='g', linestyle='--', linewidth=2, label=f'Lambda Max (Noise Limit): {lambda_max:.2f}')
    
    # Highlight Significant Eigenvalues
    sig_evals = evals[evals > lambda_max]
    plt.scatter(sig_evals, np.zeros_like(sig_evals), color='red', s=50, zorder=5, label=f'Significant Signals ({len(sig_evals)})')
    
    plt.title(title)
    plt.xlabel('Eigenvalue')
    plt.ylabel('Density Probability')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print(f"--- {title} ---")
    print(f"Q (T/N): {Q:.2f}")
    print(f"Lambda Min: {lambda_min:.4f}")
    print(f"Lambda Max: {lambda_max:.4f}")
    print(f"Total Eigenvalues: {len(evals)}")
    print(f"Significant Eigenvalues (> Lambda Max): {len(sig_evals)}")
    print(f"% Signal Information: {len(sig_evals)/len(evals)*100:.1f}%")
    return sig_evals

# Jalankan tes ini pada seluruh dataset untuk gambaran umum
_ = plot_rmt_test(returns, title="RMT Significance Test (Full Dataset 2023-2025)")

## 4. New Engines: Network Markowitz (RMT + MST + Centrality)

In [6]:
def get_rmt_cleaned_cov(returns_window):
    """
    Membersihkan matriks kovarians menggunakan Random Matrix Theory (Marcenko-Pastur).
    """
    T, N = returns_window.shape
    if T < N: return returns_window.cov() * 252 # Fallback jika data kurang
    
    corr_mat = returns_window.corr()
    cov_mat = returns_window.cov() * 252
    
    # Eigenvalue Decomposition
    evals, evecs = np.linalg.eigh(corr_mat)
    
    # Marcenko-Pastur Threshold
    Q = T / N
    lambda_max = (1 + 1.0/Q + 2 * np.sqrt(1.0/Q))
    
    # Filtering: Ganti eigenvalue noise (< lambda_max) dengan rata-rata noise
    noise_evals = evals[evals < lambda_max]
    if len(noise_evals) > 0:
        mean_noise = np.mean(noise_evals)
        evals[evals < lambda_max] = mean_noise
    
    # Reconstruct Correlation Matrix
    corr_clean = evecs @ np.diag(evals) @ evecs.T
    np.fill_diagonal(corr_clean, 1.0) # Pastikan diagonal = 1
    
    # Reconstruct Covariance Matrix
    std_devs = np.sqrt(np.diag(cov_mat))
    cov_clean = corr_clean * np.outer(std_devs, std_devs)
    
    return pd.DataFrame(cov_clean, index=cov_mat.index, columns=cov_mat.columns)

def get_network_centrality_weights(returns_window):
    """
    Menghitung bobot portofolio berdasarkan Network Centrality pada MST.
    Strategi: Diversifikasi ke aset 'pinggiran' (Peripheral) -> Bobot ~ 1 / Centrality
    """
    corr = returns_window.corr().abs().fillna(0)
    # Jarak Metrik untuk MST: d = sqrt(2(1-rho))
    dist = np.sqrt(2 * (1 - corr))
    
    G = nx.Graph()
    assets = returns_window.columns
    
    for i in range(len(assets)):
        for j in range(i + 1, len(assets)):
            G.add_edge(assets[i], assets[j], weight=dist.iloc[i, j])
            
    # Minimum Spanning Tree
    mst = nx.minimum_spanning_tree(G)
    
    # Hitung Centrality (Eigenvector Centrality sebagai proxy pengaruh global)
    try:
        cent = nx.eigenvector_centrality(mst, max_iter=1000)
    except:
        cent = nx.degree_centrality(mst) # Fallback
        
    # Inverse Centrality Weighting (Pilih aset yang kurang terhubung/unik)
    inv_cent = {k: 1.0/(v + 1e-6) for k,v in cent.items()}
    total_inv = sum(inv_cent.values())
    weights = {k: v/total_inv for k,v in inv_cent.items()}
    
    return weights

def optimize_network_markowitz_proposisi(returns_window, gamma=0.5):
    """
    Proposisi Network Markowitz:
    W_final = (1 - gamma) * W_Markowitz(RMT Cleaned) + gamma * W_Network(MST Centrality)
    """
    # 1. RMT Cleaned Markowitz Weights
    cov_rmt = get_rmt_cleaned_cov(returns_window)
    w_markowitz = optimize_markowitz_robust(returns_window, cov_matrix=cov_rmt)
    
    # 2. Network Centrality Weights
    w_network = get_network_centrality_weights(returns_window)
    
    # 3. Combine with Gamma
    final_weights = {}
    assets = returns_window.columns
    
    for asset in assets:
        wm = w_markowitz.get(asset, 0)
        wn = w_network.get(asset, 0)
        final_weights[asset] = (1 - gamma) * wm + gamma * wn
        
    # Normalize (just in case)
    total_w = sum(final_weights.values())
    if total_w > 0:
        final_weights = {k: v/total_w for k, v in final_weights.items()}
        
    return final_weights

def run_simulation_network_markowitz(test_dates, returns, gamma=0.5):
    # Menggunakan Top 15 Momentum sebagai Universe Awal (Sesuai Benchmark)
    start_date = test_dates[0]
    loc_idx = returns.index.get_loc(start_date)
    pre_window = returns.iloc[max(0, loc_idx-60):loc_idx]
    
    # 1. Pilih Top 15 Assets by Momentum (Sama seperti benchmark statis)
    top15_momentum = pre_window.mean().sort_values(ascending=False).head(15).index
    selected_returns = pre_window[top15_momentum]
    
    # 2. Hitung Bobot Proposisi (Sekali di awal tahun 2025 - Static Hold)
    # Asumsi: Strategi ini statis seperti benchmark 'Static Markowitz'
    # Jika ingin dinamis, loop harian bisa ditambahkan.
    weights = optimize_network_markowitz_proposisi(selected_returns, gamma=gamma)
    
    val = 100.0; history = [val]; dates = [start_date]
    for i, date in enumerate(test_dates[:-1]):
        next_date = test_dates[i+1]
        day_ret = sum(w * (returns.loc[next_date, asset] if next_date in returns.index else 0) for asset, w in weights.items())
        val *= (1 + day_ret); history.append(val); dates.append(next_date)
    
    print(f"✓ Optimized Network weights (Gamma={gamma}):")
    print({k: round(v, 3) for k, v in weights.items() if v > 0.01})
    return pd.DataFrame({'Portfolio_Value': history}, index=dates)

## 5. Run Comparative Analysis (2025)

In [7]:
test_dates_2025 = returns.loc[returns.index.year == 2025].index
results = {}

print("--- EXISTING MODELS ---")
print("Simulating Top 15 Static Markowitz (Benchmark)...")
results['Static Markowitz (Top 15)'] = run_simulation_top15_static_markowitz(test_dates_2025, returns)

print("Simulating AI-Cls (Top 15 Momentum)...")
results['AI-Cls (Top 15)'] = run_simulation_ai_momentum(test_dates_2025, returns, all_probs, opt_t, market_index, 15, 'cluster')

print("\n--- NEW NETWORK MODELS (PROPOSISI) ---")
gammas = [0, 0.5, 1]
for g in gammas:
    label = f"Network Markowitz (γ={g})"
    print(f"Simulating {label}...")
    results[label] = run_simulation_network_markowitz(test_dates_2025, returns, gamma=g)

print("\n✓ All simulations completed. Proceeding to evaluation.")

## 7. Comprehensive Performance Evaluation (Thesis Metrics)

Evaluasi kinerja model menggunakan 6 metrik utama:
1. **Sharpe Ratio (SR)**: Efficiency (Return/Risk)
2. **Mean Daily Return ($\mu$)**: Profitability
3. **Standard Deviation ($\sigma$)**: Volatility
4. **Maximum Drawdown (MDD)**: Downside Risk
5. **Value at Risk (VaR 95%)**: Tail Risk
6. **Rachev Ratio (95%)**: Tail Reward vs Tail Loss

In [8]:
def calculate_rachev_ratio(returns, alpha=0.95):
    """
    Rachev Ratio = Expected Tail Return (upper alpha%) / abs(Expected Tail Loss (lower 1-alpha%))
    """
    var_lower = np.percentile(returns, (1-alpha)*100)
    etl_lower = returns[returns <= var_lower].mean()
    
    var_upper = np.percentile(returns, alpha*100)
    etr_upper = returns[returns >= var_upper].mean()
    
    if np.abs(etl_lower) < 1e-6: return np.nan # Avoid division by zero
    return etr_upper / np.abs(etl_lower)

def evaluate_portfolio(portfolio_series, name="Strategy"):
    daily_rets = portfolio_series.pct_change().dropna()
    
    # 1. Financial Metrics
    mean_ret = daily_rets.mean()
    std_dev = daily_rets.std()
    ann_ret = (1 + mean_ret)**252 - 1
    ann_vol = std_dev * np.sqrt(252)
    sharpe = (mean_ret / std_dev) * np.sqrt(252) if std_dev > 0 else 0
    
    # 2. Risk Metrics
    cum_ret = (1 + daily_rets).cumprod()
    peak = cum_ret.cummax()
    drawdown = (cum_ret - peak) / peak
    mdd = drawdown.min()
    var_95 = np.percentile(daily_rets, 5)
    
    # 3. Tail Metrics
    rachev_95 = calculate_rachev_ratio(daily_rets, 0.95)
    
    return {
        "Strategy": name,
        "Sharpe Ratio": sharpe,
        "Mean Daily Ret (%)": mean_ret * 100,
        "Daily Std Dev (%)": std_dev * 100,
        "Max Drawdown (%)": mdd * 100,
        "VaR (95%) (%)": var_95 * 100,
        "Rachev Ratio (95%)": rachev_95
    }

eval_data = []
for name, res_df in results.items():
    metrics = evaluate_portfolio(res_df['Portfolio_Value'], name)
    eval_data.append(metrics)

eval_df = pd.DataFrame(eval_data).set_index("Strategy")
eval_df = eval_df.sort_values("Sharpe Ratio", ascending=False)

print("\n=== THESIS PERFORMANCE METRICS ===")
print(eval_df.round(4))

# Visualization of Metrics
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics_to_plot = ['Sharpe Ratio', 'Mean Daily Ret (%)', 'Daily Std Dev (%)', 
                   'Max Drawdown (%)', 'VaR (95%) (%)', 'Rachev Ratio (95%)']

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i//3, i%3]
    sns.barplot(x=eval_df.index, y=eval_df[metric], ax=ax, palette="viridis")
    ax.set_title(metric)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_xlabel("")

plt.tight_layout()
plt.savefig('thesis_metrics_evaluation.png')
plt.show()